# VivaJot test 3: snapshot chosen by false alarms

The deployed recipe exactly (1500 ms, 1500 ms, negative weight 20, 10,000 steps, music
behind the word, same chatter, same snapshot choice), with THREE changes from deployed: about 900 synthetic speakers
instead of 18 Piper voices (test 1), 5,000 clips instead of 900 (test 2), and, new in this test, the
saved snapshot is the one with the fewest false alarms on 2 hours of held-out talk instead of the
first one (test 2 heard well but false-woke 77/h: its snapshot was chosen with no regard to false alarms).

Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all. About an hour. No Google Drive.
The last cell prints the bench against the deployed model and a PASS or FAIL.


In [ ]:
# Settings
PHRASE = 'veeva jaht'          # the pronunciation Cary chose; one phrase only
NAME = 'vivajot'
CLIP_MS = 1500                 # same as the deployed model
NEG_WEIGHT = 20
STEPS = 10000
SAMPLES = 5000                 # test 2: more clips (deployed and test 1 had 900)
FALSE_WAKE_LIMIT = 1.0         # per hour of continuous talk

import os, json, glob, subprocess, sys
os.chdir('/content')
OUT = '/content/out'
os.makedirs(OUT, exist_ok=True)


In [ ]:
# Install. Pinned to the commit this notebook was written against.
# datasets 2.21.0: newer versions cannot decode audio without torchcodec.
%pip install -q 'datasets==2.21.0'
%pip install -q 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
if not os.path.exists('microWakeWord'):
    subprocess.run('git clone https://github.com/kahrendt/microWakeWord && git -C microWakeWord checkout 4665173', shell=True, check=True)
%pip install -q -e ./microWakeWord
%pip install -q piper-tts   # Colab is on Python 3.13; piper-phonemize-cross has no build for it and is not needed
if not os.path.exists('piper-sample-generator'):
    subprocess.run('git clone https://github.com/rhasspy/piper-sample-generator', shell=True, check=True)
pt = 'piper-sample-generator/models/en_US-libritts_r-medium.pt'
if not os.path.exists(pt):
    subprocess.run(['wget', '-q', '-O', pt, 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'], check=True)

# microWakeWord stride assertion fails on some clip lengths
p = 'microWakeWord/microwakeword/utils.py'
s = open(p).read()
old = '            assert spectrogram.shape[0] % stride == 0'
new = """            remainder = spectrogram.shape[0] % stride
            if remainder != 0:
                spectrogram = spectrogram[:spectrogram.shape[0] - remainder]"""
open(p, 'w').write(s.replace(old, new))

import importlib, site
importlib.invalidate_caches()
site.main()
import datasets, microwakeword
assert datasets.__version__ == '2.21.0', 'Runtime -> Restart session, then Run all again'
print('installed')

In [ ]:
%%writefile bench.py
"""VivaJot wake word bench: compare models on the same audio, the way the speaker detects.

Reports, for every model and cutoff:
  - false wakes per hour on continuous talk audio
  - recall on "VivaJot" at several voice levels, from normal to very soft
and picks the most sensitive cutoff that stays inside the false-wake limit.

Mirrors ESPHome micro_wake_word (streaming_model.cpp, micro_wake_word.cpp):
  - features scaled to int8 with the firmware's integer math
  - quantized cutoff int(cutoff * 255); fires when the sum of the last 5 raw outputs > cutoff * 5
  - VAD (stock vad.tflite, cutoff 0.5, window 5) must also pass, or the wake is blocked
  - after a wake, 100 10 ms slices below cutoff before the next one can fire

Run on Colab or locally:
  python bench.py --model deployed=vivajot.tflite --model new=new.tflite --vad vad.tflite \
      --talk talk_audio/ --pos positives/ --out results/
  python bench.py --selftest
"""
import argparse
import glob
import json
import os

import numpy as np

CUTOFFS = [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 0.97]
WINDOW = 5
COOLDOWN_SLICES = 100
STRIDE = 3
VAD_CUTOFF = int(0.5 * 255)
# Peak level at the model's input. 300 is about a quiet answer in the hub's STT log (peak 314).
LEVELS = {'normal': 6000, 'quiet': 1500, 'soft': 400, 'very_soft': 120}
NOISE_RMS = 30          # room noise floor, about -60 dBFS
ROOM_SEEDS = (1, 2, 3)  # the same three rooms for every model
FALSE_WAKE_LIMIT = 1.0  # per hour of continuous talk; a few a day with the TV on


def load_interpreter(path):
    try:
        from ai_edge_litert.interpreter import Interpreter
    except ImportError:
        from tensorflow.lite.python.interpreter import Interpreter
    i = Interpreter(model_path=path)
    i.allocate_tensors()
    return i


def load_audio(path):
    import soundfile as sf
    d, sr = sf.read(path, dtype='float32', always_2d=True)
    d = d.mean(axis=1)
    if sr != 16000:
        if sr > 16000:  # anti-alias below 8 kHz before interpolating
            t = np.arange(-40, 41)
            fir = np.sinc(2 * 7600 / sr * t) * np.hamming(len(t))
            d = np.convolve(d, fir / fir.sum(), mode='same')
        d = np.interp(np.arange(int(len(d) * 16000 / sr)) * sr / 16000, np.arange(len(d)), d).astype(np.float32)
    return (np.clip(d, -1, 1) * 32767).astype(np.int16)


def features(pcm):
    """int8 features exactly as the firmware feeds them to the model, one row per 10 ms."""
    from pymicro_features import MicroFrontend
    fe = MicroFrontend()
    raw = pcm.tobytes()
    out, pos = [], 0
    while pos + 320 < len(raw):
        r = fe.process_samples(raw[pos:pos + 320])
        pos += r.samples_read * 2
        if r.features:
            out.append(r.features)
    if not out:
        return np.zeros((0, 40), dtype=np.int8)
    # pymicro_features returns value * 0.0390625; undo that to get the frontend's integers.
    vals = np.rint(np.array(out, dtype=np.float64) / 0.0390625).astype(np.int32)
    q = (vals * 256 + 333) // 666 - 128
    return np.clip(q, -128, 127).astype(np.int8)


class Stream:
    def __init__(self, path):
        self.i = load_interpreter(path)
        self.inp = self.i.get_input_details()[0]
        self.out = self.i.get_output_details()[0]

    def run(self, feats):
        """Raw uint8 outputs, one per 30 ms, from a fresh streaming state."""
        self.i.reset_all_variables()
        n = len(feats) // STRIDE
        res = np.zeros(n, dtype=np.uint8)
        for k in range(n):
            self.i.set_tensor(self.inp['index'], feats[k * STRIDE:(k + 1) * STRIDE].reshape(self.inp['shape']))
            self.i.invoke()
            res[k] = self.i.get_tensor(self.out['index']).flat[0]
        return res


def detect(probs, vad, cutoff):
    """Indices of inferences that wake the speaker at this cutoff. vad=None skips the VAD gate."""
    c = int(cutoff * 255)
    buf = [0] * WINDOW
    slot = 0
    ignore = -COOLDOWN_SLICES
    last = 0
    vbuf = [0] * WINDOW
    hits = []
    for k, p in enumerate(probs):
        p = int(p)
        slot = (slot + 1) % WINDOW
        ignore = min(ignore + 2 * (last < c), 0)  # the two slices before this inference
        buf[slot] = p
        last = p
        if p < c:
            ignore = min(ignore + 1, 0)
        if vad is None:
            vad_on = True
        else:
            vbuf[slot] = int(vad[k])
            vad_on = sum(vbuf) > VAD_CUTOFF * WINDOW
        if ignore >= 0 and sum(buf) > c * WINDOW and vad_on:
            hits.append(k)
            buf = [0] * WINDOW
            last = 0
            ignore = -COOLDOWN_SLICES
    return hits


def room_impulse(seed, rt60=0.45, direct=0.35):
    """Small synthetic room, so every model hears exactly the same reverb."""
    rng = np.random.default_rng(seed)
    n = int(rt60 * 16000)
    tail = rng.normal(size=n) * np.exp(-6.9 * np.arange(n) / n)
    ir = (1 - direct) * tail / np.sqrt((tail ** 2).sum())
    ir[0] += direct
    return ir


def place_in_room(pcm, peak, seed, pad_s=1.5):
    """Render a clip at the mic: room reverb, a set peak level, and the room's noise floor."""
    rng = np.random.default_rng(seed + 1000)
    ir = room_impulse(seed)
    n = len(pcm) + len(ir) - 1
    x = np.fft.irfft(np.fft.rfft(pcm.astype(np.float64), n) * np.fft.rfft(ir, n), n)
    x *= peak / max(np.abs(x).max(), 1.0)
    pad = int(pad_s * 16000)
    x = np.concatenate([np.zeros(pad), x, np.zeros(pad)])
    x += rng.normal(scale=NOISE_RMS, size=len(x))
    return np.clip(np.rint(x), -32768, 32767).astype(np.int16)


def false_wakes(streams, vad_stream, files, max_hours=None):
    """{model: {cutoff: wakes per hour}} plus the hours of audio and each wake's file and time."""
    counts = {m: {c: 0 for c in CUTOFFS} for m in streams}
    times = {m: [] for m in streams}
    secs = 0.0
    for f in files:
        feats = features(load_audio(f))
        vad = vad_stream.run(feats)
        for m, s in streams.items():
            probs = s.run(feats)
            for c in CUTOFFS:
                hits = detect(probs, vad, c)
                counts[m][c] += len(hits)
                if c == 0.90:
                    times[m] += [(os.path.basename(f), round(h * 0.03, 1)) for h in hits]
        secs += len(feats) * 0.01
        if max_hours and secs >= max_hours * 3600:
            break
    hours = secs / 3600
    faph = {m: {c: counts[m][c] / max(hours, 1e-9) for c in CUTOFFS} for m in streams}
    return faph, hours, times


def recall(streams, vad_stream, files, levels=LEVELS):
    """{model: {level: {cutoff: fraction of clips that woke}}}. Each clip is heard in every room."""
    got = {m: {l: {c: 0 for c in CUTOFFS} for l in levels} for m in streams}
    n = 0
    for f in files:
        pcm = load_audio(f)
        for seed in ROOM_SEEDS:
            n += 1
            for l, peak in levels.items():
                feats = features(place_in_room(pcm, peak, seed))
                vad = vad_stream.run(feats)
                for m, s in streams.items():
                    probs = s.run(feats)
                    for c in CUTOFFS:
                        got[m][l][c] += bool(detect(probs, vad, c))
    return {m: {l: {c: v / max(n, 1) for c, v in d.items()} for l, d in lv.items()} for m, lv in got.items()}, n


def pick_cutoff(faph_by_cutoff, limit=FALSE_WAKE_LIMIT):
    """Most sensitive cutoff inside the limit: keep real calls, accept occasional false wakes."""
    ok = [c for c in CUTOFFS if faph_by_cutoff[c] <= limit]
    return min(ok) if ok else None


def mine(stream, vad_stream, path, cutoff=0.5, seconds=3.0):
    """Hard negatives: the float spectrogram leading up to each wake, in training's scale."""
    pcm = load_audio(path)
    feats = features(pcm)
    hits = detect(stream.run(feats), vad_stream.run(feats), cutoff)
    n = int(seconds * 100)
    spec = (feats.astype(np.float32) + 128) * 0.1019607857
    return [spec[max(0, h * STRIDE - n):h * STRIDE] for h in hits if h * STRIDE >= n]


def report(faph, hours, rec, n_clips):
    lines = [f'# Wake word bench\n\nTalk audio: {hours:.2f} h. Clips: {n_clips} (clip x room).\n',
             '## False wakes per hour on talk\n',
             '| Model | ' + ' | '.join(f'{c:.2f}' for c in CUTOFFS) + ' |',
             '|---|' + '---|' * len(CUTOFFS)]
    for m, d in faph.items():
        lines.append(f'| {m} | ' + ' | '.join(f'{d[c]:.1f}' for c in CUTOFFS) + ' |')
    for level in LEVELS:
        lines += [f'\n## Recall, {level} voice\n',
                  '| Model | ' + ' | '.join(f'{c:.2f}' for c in CUTOFFS) + ' |',
                  '|---|' + '---|' * len(CUTOFFS)]
        for m in rec:
            lines.append(f'| {m} | ' + ' | '.join(f'{rec[m][level][c] * 100:.0f}%' for c in CUTOFFS) + ' |')
    lines += [f'\n## Best setting (false wakes <= {FALSE_WAKE_LIMIT}/h)\n',
              '| Model | Cutoff | False wakes/h | ' + ' | '.join(LEVELS) + ' |',
              '|---|---|---|' + '---|' * len(LEVELS)]
    for m in faph:
        c = pick_cutoff(faph[m])
        if c is None:
            lines.append(f'| {m} | none meets the limit | | ' + ' | '.join('' for _ in LEVELS) + ' |')
        else:
            lines.append(f'| {m} | {c:.2f} | {faph[m][c]:.1f} | '
                         + ' | '.join(f'{rec[m][l][c] * 100:.0f}%' for l in LEVELS) + ' |')
    return '\n'.join(lines) + '\n'


def selftest():
    # One long peak fires once; the buffer resets and it must drop below cutoff for 1 s to refire.
    assert detect([0] * 40 + [255] * 20 + [0] * 40, None, 0.9) == [44]
    # Average just under cutoff never fires, even with single outputs at 100%.
    assert detect([0] * 40 + [255, 200] * 40, None, 0.95) == []
    # No wake during the start-up second.
    assert detect([255] * 20, None, 0.9) == []
    # VAD off blocks the wake; VAD on lets it through.
    p = [0] * 40 + [255] * 10
    assert detect(p, [0] * 50, 0.9) == []
    assert detect(p, [255] * 50, 0.9) == [44]
    # Two wakes need 100 slices (34 inferences) below cutoff between them.
    assert len(detect([0] * 40 + [255] * 5 + [0] * 40 + [255] * 5, None, 0.9)) == 2
    assert len(detect([0] * 40 + [255] * 5 + [0] * 20 + [255] * 5, None, 0.9)) == 1
    # cutoff quantizes like the firmware
    assert int(0.90 * 255) == 229
    assert pick_cutoff({c: (0.5 if c >= 0.8 else 9.0) for c in CUTOFFS}) == 0.80
    print('selftest OK')


def audio_files(d):
    return sorted(f for e in ('wav', 'flac', 'mp3', 'ogg') for f in glob.glob(os.path.join(d, '**', '*.' + e), recursive=True))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--model', action='append', default=[], help='name=path.tflite')
    ap.add_argument('--vad', default='vad.tflite')
    ap.add_argument('--talk', help='folder of continuous talk audio')
    ap.add_argument('--pos', help='folder of "VivaJot" clips')
    ap.add_argument('--out', default='bench_results')
    ap.add_argument('--max-hours', type=float)
    ap.add_argument('--selftest', action='store_true')
    a = ap.parse_args()
    if a.selftest:
        return selftest()
    streams = {n: Stream(p) for n, p in (m.split('=', 1) for m in a.model)}
    vad = Stream(a.vad)
    faph, hours, times = false_wakes(streams, vad, audio_files(a.talk), a.max_hours)
    rec, n = recall(streams, vad, audio_files(a.pos))
    os.makedirs(a.out, exist_ok=True)
    text = report(faph, hours, rec, n)
    with open(os.path.join(a.out, 'bench.md'), 'w', encoding='utf-8') as f:
        f.write(text)
    with open(os.path.join(a.out, 'bench.json'), 'w', encoding='utf-8') as f:
        json.dump({'hours': hours, 'clips': n, 'false_wakes_per_hour': faph, 'recall': rec, 'wakes_at_0.90': times}, f, indent=1)
    print(text)


if __name__ == '__main__':
    main()

In [ ]:
# Download helpers. Every folder is counted, so an empty download stops the run here.
import io, urllib.request
import numpy as np, soundfile as sf, pyarrow.parquet as pq
from huggingface_hub import hf_hub_download, list_repo_files

def need(folder, minimum):
    n = len(glob.glob(folder + '/**/*.wav', recursive=True))
    assert n >= minimum, f'{folder}: {n} files, expected at least {minimum}'
    print(folder, n, 'files')

def rows(repo, filename):
    path = hf_hub_download(repo, filename, repo_type='dataset')
    f = pq.ParquetFile(path)
    for batch in f.iter_batches(batch_size=64):
        yield from batch.to_pylist()
    os.remove(os.path.realpath(path))

def to16k(row):
    x, sr = sf.read(io.BytesIO(row['audio']['bytes']), dtype='float32', always_2d=True)
    x = x.mean(axis=1)
    if sr != 16000:
        x = np.interp(np.arange(int(len(x) * 16000 / sr)) * sr / 16000, np.arange(len(x)), x)
    return x

def long_files(source, out, minutes=10):
    """Joins short clips into long files, so training and mining see continuous talk."""
    os.makedirs(out, exist_ok=True)
    buf, size, n = [], 0, len(glob.glob(out + '/*.wav'))
    for x in source:
        buf.append(x); size += len(x)
        if size >= minutes * 960000:
            sf.write(f'{out}/{n:05d}.wav', np.concatenate(buf), 16000, subtype='PCM_16')
            buf, size, n = [], 0, n + 1
    if size:
        sf.write(f'{out}/{n:05d}.wav', np.concatenate(buf), 16000, subtype='PCM_16')

def shard_rows(repo, files, keep=lambda r: True):
    for f in files:
        for r in rows(repo, f):
            if keep(r):
                try:
                    yield to16k(r)
                except Exception:
                    pass

In [ ]:
# Talk audio the models are judged on. Neither model has trained on it.
PS = 'MLCommons/peoples_speech'
ps_files = sorted(f for f in list_repo_files(PS, repo_type='dataset') if f.startswith('clean/train-'))
if not glob.glob('talk_bench/*.wav'):
    long_files(shard_rows(PS, ps_files[-1:]), 'talk_bench')
need('talk_bench', 10)

# Held-out talk for choosing the snapshot during training: a different shard from the bench, about 2 hours.
import itertools
if not glob.glob('talk_val/*.wav'):
    long_files(itertools.islice(shard_rows(PS, ps_files[-2:-1]), 700), 'talk_val')
need('talk_val', 5)


In [ ]:
# Room echo and music behind the word, the same sources the deployed model was trained with.
from huggingface_hub import snapshot_download
if not glob.glob('rirs/*.wav'):
    os.makedirs('rirs', exist_ok=True)
    d = snapshot_download('davidscripka/MIT_environmental_impulse_responses', repo_type='dataset')
    for f in glob.glob(d + '/16khz/*.wav'):
        os.symlink(os.path.realpath(f), 'rirs/' + os.path.basename(f))
need('rirs', 250)

if not glob.glob('fma_16k/*.wav'):
    os.makedirs('fma', exist_ok=True)
    subprocess.run('cd fma && wget -q -O fma_xs.zip https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip && unzip -q fma_xs.zip', shell=True, check=True)
    os.makedirs('fma_16k', exist_ok=True)
    for i, mp3 in enumerate(sorted(glob.glob('fma/**/*.mp3', recursive=True))):
        subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', mp3, '-ac', '1', '-ar', '16000', f'fma_16k/{i:05d}.wav'])
need('fma_16k', 150)


In [ ]:
# "VivaJot" samples: about 900 different synthetic speakers, several paces. THE one change under test.
def piper(text, model, n, out, batch=100, extra=()):
    r = subprocess.run([sys.executable, '-m', 'piper_sample_generator', text, '--model', model, '--max-samples', str(n),
                        '--batch-size', str(batch), '--output-dir', os.path.abspath(out), *extra],
                       cwd='piper-sample-generator', capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-1500:]

voices = json.load(urllib.request.urlopen('https://huggingface.co/rhasspy/piper-voices/resolve/main/voices.json'))
def get_voice(key):
    for path in voices[key]['files']:
        if path.endswith('.onnx') or path.endswith('.onnx.json'):
            local = 'voices/' + os.path.basename(path)
            if not os.path.exists(local):
                os.makedirs('voices', exist_ok=True)
                urllib.request.urlretrieve('https://huggingface.co/rhasspy/piper-voices/resolve/main/' + path, local)
    return os.path.abspath('voices/' + key + '.onnx')

if not glob.glob('pos_all/*.wav'):
    piper(PHRASE, 'models/en_US-libritts_r-medium.pt', SAMPLES, 'pos_all',
          extra=('--max-speakers', '877', '--length-scales', '0.9', '1.0', '1.0', '1.15', '1.3', '1.5',
                 '--noise-scales', '0.667', '0.8', '1.0'))
need('pos_all', SAMPLES)


In [ ]:
# Turn audio into features the model trains on. Same settings as before, with the noise present.
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap

os.makedirs('negative_datasets', exist_ok=True)
for name in ['dinner_party', 'no_speech', 'speech']:
    if not os.path.exists('negative_datasets/' + name):
        url = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/' + name + '.zip'
        subprocess.run(f'wget -q {url} && unzip -q {name}.zip && rm {name}.zip', shell=True, check=True, cwd='negative_datasets')
    assert glob.glob(f'negative_datasets/{name}/training/*_mmap'), name

def mmap(gen, out):
    os.makedirs(out, exist_ok=True)
    RaggedMmap.from_generator(out_dir=out + '/wakeword_mmap', sample_generator=gen, batch_size=100, verbose=True)

if not os.path.exists('f_pos/testing/wakeword_mmap'):
    clips = Clips(input_directory='pos_all', file_pattern='*.wav', max_clip_duration_s=None, remove_silence=False,
                  random_split_seed=10, split_count=0.1)
    aug = Augmentation(augmentation_duration_s=3.2,
                       augmentation_probabilities={'SevenBandParametricEQ': 0.1, 'TanhDistortion': 0.1, 'PitchShift': 0.1,
                                                   'BandStopFilter': 0.1, 'AddColorNoise': 0.1, 'AddBackgroundNoise': 0.75,
                                                   'Gain': 1.0, 'RIR': 0.5},
                       impulse_paths=['rirs'], background_paths=['fma_16k'],
                       background_min_snr_db=-5, background_max_snr_db=10, min_jitter_s=0.195, max_jitter_s=0.205)
    for split, name, rep, slide in [('training', 'train', 2, 10), ('validation', 'validation', 1, 10), ('testing', 'test', 1, 1)]:
        g = SpectrogramGeneration(clips=clips, augmenter=aug, slide_frames=slide, step_ms=10)
        mmap(g.spectrogram_generator(split=name, repeat=rep), f'f_pos/{split}')

def negative(folder, out, seconds=None):
    if os.path.exists(out + '/wakeword_mmap'):
        return
    clips = Clips(input_directory=folder, file_pattern='**/*.wav', max_clip_duration_s=None, remove_silence=False,
                  random_split_seed=10, split_count=0.01)
    g = SpectrogramGeneration(clips=clips, augmenter=None, step_ms=10, split_spectrogram_duration_s=seconds)
    mmap(g.spectrogram_generator(split='train'), out)

negative('talk_val', 'f_ambient/validation_ambient')
print('features ready')


In [ ]:
# Training as before. THE change under test: the snapshot kept is the one with the fewest false alarms
# per hour on held-out talk (f_ambient), and only among those, the best hearing.
import yaml

feats = [
    dict(features_dir='f_pos', sampling_weight=2.0, penalty_weight=1.0, truth=True, truncation_strategy='truncate_start', type='mmap'),
    dict(features_dir='negative_datasets/speech', sampling_weight=10.0, penalty_weight=1.0, truth=False, truncation_strategy='random', type='mmap'),
    dict(features_dir='negative_datasets/dinner_party', sampling_weight=10.0, penalty_weight=1.0, truth=False, truncation_strategy='random', type='mmap'),
    dict(features_dir='negative_datasets/no_speech', sampling_weight=5.0, penalty_weight=1.0, truth=False, truncation_strategy='random', type='mmap'),
    dict(features_dir='f_ambient', sampling_weight=0.0, penalty_weight=1.0, truth=False, truncation_strategy='split', type='mmap'),
]
cfg = dict(window_step_ms=10, train_dir='trained_models/' + NAME, features=feats, training_steps=[STEPS],
           positive_class_weight=[1], negative_class_weight=[NEG_WEIGHT], learning_rates=[0.001], batch_size=128,
           time_mask_max_size=[0], time_mask_count=[0], freq_mask_max_size=[0], freq_mask_count=[0],
           eval_step_interval=1000, clip_duration_ms=CLIP_MS, target_minimization=FALSE_WAKE_LIMIT,
           minimization_metric='ambient_false_positives_per_hour', maximization_metric='average_viable_recall')
yaml.dump(cfg, open('training_parameters.yaml', 'w'))

TRAIN = """python -m microwakeword.model_train_eval --training_config=training_parameters.yaml --train 1 --restore_checkpoint 1 \
--test_tf_nonstreaming 0 --test_tflite_nonstreaming 0 --test_tflite_nonstreaming_quantized 0 --test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 --use_weights best_weights mixednet --pointwise_filters "64,64,64,64" \
--repeat_in_block "1, 1, 1, 1" --mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' --residual_connection "0,0,0,0" \
--first_conv_filters 32 --first_conv_kernel_size 5 --stride 3"""

MODEL = f'trained_models/{NAME}/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'
if not os.path.exists(MODEL):
    p = subprocess.Popen(TRAIN, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
    lines = []
    for line in p.stdout:
        if 'Validation Batch' in line:
            continue
        lines.append(line)
        if 'nonstreaming' in line or 'So far the best' in line or 'Step #' in line and line.strip().endswith('000:'):
            print(line.strip()[:220])
    text = ''.join(lines)
    open(OUT + '/training.log', 'w').write(text)
    assert p.wait() == 0, text[-3000:]
    import re
    checks = re.findall(r'ambient false positives = (\d+), estimated false positives per hour = ([0-9.]+)', text)
    print('false alarms per hour on held-out talk at each check:', [f for _, f in checks])
    assert len(checks) >= 3, 'training did not report false alarms on held-out talk'
assert os.path.exists(MODEL), 'no model was exported'
print('trained', os.path.getsize(MODEL), 'bytes')


In [ ]:
# Final comparison: new model against the deployed model on the same audio, the way the speaker detects.
# Test voices are British Piper voices that neither model trained on.
import wave, shutil, base64
urllib.request.urlretrieve('https://github.com/esphome/micro-wake-word-models/raw/main/models/v2/vad.tflite', 'vad.tflite')
urllib.request.urlretrieve('https://raw.githubusercontent.com/saas-maker/HomeAssistant/master/wakewords/vivajot.tflite', 'deployed.tflite')
import bench
vad = bench.Stream('vad.tflite')
from piper import PiperVoice
from piper.config import SynthesisConfig

GB = ['en_GB-alan-medium', 'en_GB-cori-medium', 'en_GB-jenny_dioco-medium', 'en_GB-northern_english_male-medium']
SET = [(1.0, .667), (1.0, .9), (1.15, .667), (1.15, .9), (1.3, .667), (1.3, .9), (1.5, .667), (.9, .667)]
if not glob.glob('pos_heldout/*/*.wav'):
    for k in GB:
        v = PiperVoice.load(get_voice(k))
        os.makedirs('pos_heldout/' + k, exist_ok=True)
        for i, (ls, ns) in enumerate(SET):
            with wave.open(f'pos_heldout/{k}/{i}.wav', 'wb') as w:
                v.synthesize_wav(PHRASE, w, syn_config=SynthesisConfig(length_scale=ls, noise_scale=ns))
need('pos_heldout', 32)

models = {'deployed': bench.Stream('deployed.tflite'), 'new': bench.Stream(MODEL)}
faph, hours, times = bench.false_wakes(models, vad, bench.audio_files('talk_bench'))
rec, n = bench.recall(models, vad, bench.audio_files('pos_heldout'))
text = bench.report(faph, hours, rec, n)
cut = bench.pick_cutoff(faph['new'])
verdict = 'FAIL: no setting meets the false-wake limit'
if cut is not None:
    worse = [l for l in ('normal', 'quiet', 'soft') if rec['new'][l][cut] < rec['deployed'][l][0.90]]
    verdict = f'PASS at cutoff {cut}' if not worse else f'FAIL at cutoff {cut}: hears worse than deployed on {worse}'
text += '\n## Verdict\n\n' + verdict + '\n'
print(text)

open(OUT + '/bench.md', 'w').write(text)
json.dump({'cutoff': cut, 'verdict': verdict, 'false_wakes_per_hour': faph, 'recall': rec}, open(OUT + '/bench.json', 'w'), indent=1)
shutil.copy(MODEL, OUT + '/vivajot.tflite')
json.dump({'type': 'micro', 'wake_word': 'VivaJot', 'author': 'SaaS Factory / VivaJot', 'model': 'vivajot.tflite', 'trained_languages': ['en'],
           'version': 2, 'micro': {'probability_cutoff': cut or 0.97, 'feature_step_size': 10, 'sliding_window_size': 5,
                                   'tensor_arena_size': 45000, 'minimum_esphome_version': '2024.7.0'}}, open(OUT + '/vivajot.json', 'w'), indent=2)
print('Saved:', os.listdir(OUT))
print('TFLITE_B64_START')
print(base64.b64encode(open(MODEL, 'rb').read()).decode())
print('TFLITE_B64_END')